# 2.01 基本概念：状态、动作、奖励、回报与 MDP

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.01_basic_concepts.ipynb)


## 为什么要学

在你的真机任务里：

- **状态**：相机图像 + 关节角度；
- **动作**：下一段时间的关节目标位置（一个动作块）；
- **奖励**：线头插进去了就 +1，否则 0；
- **策略**：VLA 模型本身，$\pi(\text{动作}|\text{状态})$。

强化学习就是：**通过与环境交互得到的奖励，让策略变得更好**。本课用一个 5×5 的网格世界把这些概念全部具体化，后面所有算法都在它上面运行。对应书中第 1 章。

---

## 1. 网格世界

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld
from vla_ladder.envs.gridworld import ACTION_NAMES

env = GridWorld()
print("状态数:", env.n_states, " 动作数:", env.n_actions, " 动作:", ACTION_NAMES)
env.plot(title="grid world: blue=target, yellow=forbidden")
plt.show()

一个智能体（agent）在格子里移动：

- **状态（state）** $s$：所在的格子。共 25 个状态 $\mathcal{S}=\{s_1,\dots,s_{25}\}$（代码里编号 0–24，按行优先）。
- **动作（action）** $a$：上、右、下、左、不动，共 5 个 $\mathcal{A}=\{a_1,\dots,a_5\}$。
- **状态转移（state transition）**：执行动作后到达的新状态。撞墙会留在原地。我们的网格世界是**确定性**的，但一般情况下用条件概率 $p(s'|s,a)$ 描述。
- **奖励（reward）** $r$：
  - 撞墙 $r_{boundary}=-1$；进入禁区（黄色）$r_{forbidden}=-1$；
  - 到达目标（蓝色）$r_{target}=+1$；其它 $0$。
  一般地，用 $p(r|s,a)$ 描述。

In [ ]:
s = env.to_index((0, 0))
for a in range(5):
    ns, r = env._next(s, a)
    print(f"在 (0,0) 执行 {ACTION_NAMES[a]} -> 到达 {env.to_rc(ns)}，奖励 {r}")

---

## 2. 策略（policy）

策略告诉智能体在每个状态下怎么选动作：$\pi(a|s)$ 是在状态 $s$ 选动作 $a$ 的概率，满足 $\sum_a\pi(a|s)=1$。

- **确定性策略**：每个状态只选一个动作（某个动作概率为 1）；
- **随机性策略**：按概率分布选动作。

**表格表示**：25×5 的矩阵，第 $s$ 行是状态 $s$ 下各动作的概率。

In [ ]:
from vla_ladder.envs import policy_to_matrix
rng = np.random.default_rng(0)
deterministic = rng.integers(0, 5, size=env.n_states)          # 每个状态随便选一个动作
random_policy = np.full((env.n_states, env.n_actions), 0.2)    # 均匀随机策略
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
env.plot(policy=deterministic, title="deterministic policy", ax=ax[0])
env.plot(policy=random_policy, title="uniform random policy", ax=ax[1])
plt.show()
print("确定性策略的矩阵表示（前 3 行）:\n", policy_to_matrix(deterministic)[:3])

---

## 3. 轨迹、回报与折扣

智能体按策略与环境交互，得到一条**轨迹**：

$$
s_0 \xrightarrow{a_0} r_1, s_1 \xrightarrow{a_1} r_2, s_2 \xrightarrow{a_2} r_3, \dots
$$

（书中约定：在 $s_t$ 执行 $a_t$ 后得到的奖励记作 $r_{t+1}$。）

**折扣回报（discounted return）** 是沿轨迹的奖励加权和：

$$
G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \dots = \sum_{k=0}^{\infty}\gamma^k r_{t+k+1}
$$

**折扣因子** $\gamma\in[0,1)$ 的两个作用：

1. 数学上保证无穷和收敛（奖励有界时 $|G_t|\le\frac{r_{max}}{1-\gamma}$）；
2. 行为上：$\gamma$ 越小越"短视"，越看重眼前的奖励；$\gamma$ 越接近 1 越"远视"。

> 机器人任务通常是**回合制（episodic）**的：插好了或超时就结束。网格世界是**持续性**的：到达目标后可以一直待在那里，每步都拿 +1。书中把回合制任务也转成持续性任务来统一处理。

一个重要的递推关系（后面贝尔曼方程的来源）：

$$
G_t = r_{t+1} + \gamma G_{t+1}
$$

### 练习 1：计算折扣回报

给定奖励序列 `rewards = [r_1, r_2, ..., r_T]`，返回每个时刻的回报 `[G_0, G_1, ..., G_{T-1}]`（$T$ 之后的奖励视为 0）。**要求 O(T) 复杂度**：从后往前用 $G_t = r_{t+1} + \gamma G_{t+1}$ 递推。

> 这个函数在 REINFORCE、PPO、RECAP 中都会用到。

In [ ]:
def discounted_returns(rewards, gamma):
    """rewards: 长度 T 的列表/数组 -> 长度 T 的数组，第 t 个是 G_t"""
    # TODO: 从后往前遍历
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def discounted_returns(rewards, gamma):
    """rewards: 长度 T 的列表/数组 -> 长度 T 的数组，第 t 个是 G_t"""
    G = np.zeros(len(rewards))
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running     # G_t = r_{t+1} + γ G_{t+1}
        G[t] = running
    return G

In [ ]:
# ✅ 自测
G = discounted_returns([0, 0, 1, 1, 1], gamma=0.9)
assert np.allclose(G[-1], 1) and np.allclose(G[2], 1 + 0.9 + 0.81)
assert np.allclose(G[0], 0.81 * (1 + 0.9 + 0.81))
print("通过 ✅", G.round(3))

### 练习 2：按策略采样一条轨迹

实现 `rollout(env, policy, start, T)`：从 `start` 出发，按策略矩阵 `policy (S, A)` 采样 `T` 步，返回 `(states, actions, rewards)`。

In [ ]:
def rollout(env, policy, start, T, rng=None):
    rng = rng or np.random.default_rng()
    s = env.reset(start)
    states, actions, rewards = [], [], []
    for _ in range(T):
        # TODO: 按 policy[s] 的概率选动作 a（rng.choice），执行 env.step(a)，记录 s, a, r，更新 s
        raise NotImplementedError
    return states, actions, rewards

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def rollout(env, policy, start, T, rng=None):
    rng = rng or np.random.default_rng()
    s = env.reset(start)
    states, actions, rewards = [], [], []
    for _ in range(T):
        a = rng.choice(env.n_actions, p=policy[s])
        ns, r = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        s = ns
    return states, actions, rewards

In [ ]:
# ✅ 自测：一个"好"的确定性策略，从 (0,0) 出发，应该绕开禁区走到目标并停下
good = np.array([
    1, 1, 1, 1, 2,
    0, 0, 1, 1, 2,
    0, 3, 2, 1, 2,
    0, 1, 4, 3, 2,
    0, 1, 0, 3, 3,
])
S, A, R = rollout(env, policy_to_matrix(good), (0, 0), T=20, rng=np.random.default_rng(0))
print("走过的格子:", [env.to_rc(s) for s in S[:10]])
print("奖励:", R)
G0 = discounted_returns(R, 0.9)[0]
print(f"从 (0,0) 出发的折扣回报（截断到 20 步）: {G0:.3f}")
assert S[-1] == env.to_index(env.target) and R[-1] == 1
print("通过 ✅")
env.plot(policy=good, title="a good policy"); plt.show()

---

## 4. 马尔可夫决策过程（MDP）

把上面的要素放在一起，就是 MDP：

| 要素 | 记号 | 网格世界 |
|---|---|---|
| 状态集合 | $\mathcal{S}$ | 25 个格子 |
| 动作集合 | $\mathcal{A}(s)$ | 5 个动作 |
| 状态转移概率 | $p(s'\mid s,a)$ | 确定性 |
| 奖励概率 | $p(r\mid s,a)$ | 确定性 |
| 策略 | $\pi(a\mid s)$ | 我们要找的东西 |
| 折扣因子 | $\gamma$ | 0.9 |

**马尔可夫性**：下一个状态和奖励只取决于**当前**状态和动作，与更早的历史无关：

$$
p(s_{t+1}, r_{t+1}\mid s_t, a_t, s_{t-1}, a_{t-1},\dots) = p(s_{t+1}, r_{t+1}\mid s_t, a_t)
$$

> **真机上的马尔可夫性**：单帧图像往往不满足马尔可夫性。例如看不出物体正在往哪个方向运动，被手挡住的线头位置也看不到。所以 VLA 常常输入多帧图像、多个视角或本体状态历史。ACT 的"动作分块"在某种意义上也是在缓解非马尔可夫问题（第 5 章）。

In [ ]:
# 在代码里，MDP 的"模型"就是两个数组：
print("P 的形状 (S, A, S'):", env.P.shape, "  R 的形状 (S, A):", env.R.shape)
s, a = env.to_index((1, 0)), 1    # 在 (1,0) 向右走
print("p(s'|s,a) 非零的位置:", [env.to_rc(x) for x in np.nonzero(env.P[s, a])[0]], " 奖励:", env.R[s, a])

---

## 小结

- MDP 五要素：状态、动作、转移、奖励、折扣；策略 $\pi(a|s)$ 是要优化的对象。
- 回报 $G_t=\sum_k\gamma^kr_{t+k+1}$，满足递推 $G_t = r_{t+1}+\gamma G_{t+1}$。
- 下一课的核心问题：**如何评价一个策略有多好？**答案是"状态值"，它满足贝尔曼方程。

**思考题**：上面的好策略在禁区格子 (1,1)、(1,2) 也给了动作。这些格子"本来不该去"，为什么还要给它们定义策略？（提示：如果智能体从那里出发呢？贝尔曼方程需要**所有状态**的值。）